# AgentCore Episodic Memory - Assistente de Notas de Reunião

Este tutorial demonstra como construir um assistente de notas de reunião usando
agentes Strands integrados com AgentCore Episodic Memory. O agente aprende
com reuniões anteriores, lembrando decisões tomadas, itens de ação atribuídos e
preferências dos participantes, permitindo fornecer assistência contextualizada
em reuniões recorrentes.

Detalhes do Tutorial:
- Tipo de tutorial: Long term Episodic
- Tipo de agente: Assistente de Notas de Reunião
- Framework de agentes: Strands Agents
- Modelo LLM: Anthropic Claude Haiku 4.5
- Componentes: AgentCore Episodic Memory com Reflections

Você aprenderá a:
- Configurar o AgentCore Memory com a estratégia Episodic
- Criar memory hooks para captura automática de episódios
- Recuperar episódios e reflexões de reuniões anteriores
- Construir um agente que aprende padrões de reuniões e preferências dos participantes

## Passo 1: Instalar Dependências e Configuração

In [35]:
!pip install -qr requirements.txt

In [36]:
import logging
import json
from typing import Dict, List
from datetime import datetime
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("meeting-notes-assistant")

from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

In [37]:
# Configuration
REGION = "us-west-2"
PARTICIPANT_ID = "participant_001"
SESSION_ID = f"meeting_{datetime.now().strftime('%Y%m%d%H%M%S')}"

## Passo 2: Criar Recurso de Memória com Estratégia Episodic

A estratégia episodic automaticamente:
- Detecta quando episódios são concluídos dentro das conversas
- Extrai registros estruturados de episódios (situação, intenção, avaliação, justificativa)
- Gera reflexões que identificam padrões entre episódios

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=REGION)
memory_name = "MeetingNotesEpisodicMemory"

# Define episodic memory strategy
strategies = [
    {
        StrategyType.EPISODIC.value: {
            "name": "MeetingEpisodes",
            "description": "Captures meeting discussions and generates reflections on meeting patterns",
            "namespaceTemplates": ["meetings/actor/{actorId}/episodes"],
            "reflectionConfiguration": {
                "namespaceTemplates": ["meetings/actor/{actorId}"]  # Must be same as or prefix of episodic namespace
            }
        }
    }
]

# Create memory resource
try:
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=strategies,
        description="Episodic memory for meeting notes assistant",
        event_expiry_days=180,  # TTL for short-term memory events (STM), not for long-term episodic strategy
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using: {memory_id}")
    else:
        raise
except Exception as e:
    logger.error(f"❌ ERROR: {e}")
    raise

In [39]:
# Verify episodic strategy is configured
strategies = client.get_memory_strategies(memory_id)
print(json.dumps(strategies, indent=2, default=str))

[
  {
    "strategyId": "MeetingEpisodes-1AXgDO5YV7",
    "name": "MeetingEpisodes",
    "description": "Captures meeting discussions and generates reflections on meeting patterns",
    "configuration": {
      "type": "EPISODIC",
      "reflection": {
        "episodicReflectionConfiguration": {
          "namespaces": [
            "meetings/actor/{actorId}"
          ],
          "namespaceTemplates": [
            "meetings/actor/{actorId}"
          ]
        }
      }
    },
    "type": "EPISODIC",
    "namespaces": [
      "meetings/actor/{actorId}/episodes"
    ],
    "namespaceTemplates": [
      "meetings/actor/{actorId}/episodes"
    ],
    "createdAt": "2026-03-24 19:57:47.999000-03:00",
    "updatedAt": "2026-03-24 19:57:48.539000-03:00",
    "status": "ACTIVE",
    "memoryStrategyId": "MeetingEpisodes-1AXgDO5YV7",
    "memoryStrategyType": "EPISODIC"
  }
]


## Passo 3: Criar Ferramentas de Gerenciamento de Reuniões

In [40]:
@tool
def capture_action_item(task: str, owner: str, due_date: str) -> str:
    """Capture an action item from the meeting discussion.
    
    Args:
        task: Description of the task to be completed
        owner: Person responsible for completing the task
        due_date: When the task should be completed
    
    Returns:
        Confirmation of action item capture with details
    """
    action_items = {
        "website": "Website redesign - assigned to Sarah, due next Friday",
        "budget": "Review Q3 budget allocation - assigned to Mike, due this week",
        "presentation": "Prepare stakeholder presentation - assigned to Alex, due Monday",
        "testing": "Complete user testing for new feature - assigned to QA team, due end of sprint",
    }
    
    # Simulate action item storage
    for keyword, stored_item in action_items.items():
        if keyword in task.lower():
            return f"✅ ACTION ITEM CAPTURED:\n{stored_item}\n\nNote: {task}"
    
    return f"✅ ACTION ITEM CAPTURED:\nTask: {task}\nOwner: {owner}\nDue: {due_date}"


@tool
def identify_decision(decision: str, context: str) -> str:
    """Identify and record a key decision made during the meeting.
    
    Args:
        decision: The decision that was made
        context: Context or reasoning behind the decision
    
    Returns:
        Confirmation of decision recording with summary
    """
    decisions = {
        "budget": "Approved Q3 marketing budget increase of 15%",
        "launch": "Product launch date set for November 15th",
        "vendor": "Selected AWS as cloud infrastructure provider",
        "process": "Adopted agile sprint methodology for project management",
    }
    
    # Simulate decision recording
    for keyword, stored_decision in decisions.items():
        if keyword in decision.lower():
            return f"📌 DECISION RECORDED:\n{stored_decision}\n\nRationale: {context}"
    
    return f"📌 DECISION RECORDED:\n{decision}\n\nContext: {context}"


@tool
def summarize_discussion(topic: str, key_points: str) -> str:
    """Summarize a discussion topic with key points.
    
    Args:
        topic: The discussion topic
        key_points: Main points covered in the discussion
    
    Returns:
        Structured summary of the discussion
    """
    # Simulate discussion summarization
    return f"""📝 DISCUSSION SUMMARY:

Topic: {topic}

Key Points:
{key_points}

Next Steps: Review in next meeting"""


@tool
def track_followup(previous_item: str, status: str) -> str:
    """Track follow-up status of previous action items or decisions.
    
    Args:
        previous_item: Description of the item to follow up on
        status: Current status (completed, in-progress, blocked, pending)
    
    Returns:
        Follow-up status with details
    """
    # Simulate follow-up tracking
    statuses = {
        "completed": "✅ COMPLETED",
        "in-progress": "🔄 IN PROGRESS",
        "blocked": "🚫 BLOCKED",
        "pending": "⏳ PENDING",
    }
    
    status_emoji = statuses.get(status.lower(), "❓ UNKNOWN")
    
    return f"""{status_emoji}
Item: {previous_item}
Status: {status}
Last Updated: {datetime.now().strftime('%Y-%m-%d')}"""


logger.info("✅ Meeting management tools ready")

2026-03-24 20:00:44,404 - INFO - ✅ Meeting management tools ready


## Passo 4: Criar Hook Provider de Episodic Memory

O hook provider:
- Recupera episódios e reflexões relevantes de reuniões anteriores antes de processar consultas
- Salva interações de reuniões como eventos para extração de episódios
- Episódios são automaticamente detectados e extraídos pelo AgentCore

In [41]:
def get_namespaces(mem_client: MemoryClient, memory_id: str) -> Dict:
    """Get namespace mapping for memory strategies."""
    strategies = mem_client.get_memory_strategies(memory_id)
    result = {}
    for strategy in strategies:
        reflection_config = strategy.get("reflectionConfiguration", {})
        result[strategy["type"]] = {
            "namespaces": strategy.get("namespaces", []),
            "reflectionNamespaces": reflection_config.get("namespaces", [])
        }
    return result


class EpisodicMemoryHooks(HookProvider):
    """Memory hooks for episodic memory with reflections."""
    
    def __init__(self, memory_id: str, client: MemoryClient):
        self.memory_id = memory_id
        self.client = client
        self.namespaces = get_namespaces(self.client, self.memory_id)
    
    def retrieve_episodes_and_reflections(self, event: MessageAddedEvent):
        """Retrieve relevant episodes and reflections before processing."""
        messages = event.agent.messages
        if messages[-1]["role"] != "user" or "toolResult" in messages[-1]["content"][0]:
            return
            
        user_query = messages[-1]["content"][0]["text"]
        actor_id = event.agent.state.get("actor_id")
        
        if not actor_id:
            logger.warning("Missing actor_id in agent state")
            return
        
        try:
            all_context = []
            episodic_config = self.namespaces.get("EPISODIC", {})
            
            # Retrieve relevant episodes (indexed by "intent")
            for namespace_template in episodic_config.get("namespaces", []):
                namespace = namespace_template.format(actorId=actor_id)
                episodes = self.client.retrieve_memories(
                    memory_id=self.memory_id,
                    namespace=namespace,
                    query=user_query,  # Episodes indexed by intent
                    top_k=3
                )
                
                for episode in episodes:
                    if isinstance(episode, dict):
                        content = episode.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            if text:
                                all_context.append(f"[PAST EPISODE] {text}")
            
            # Retrieve reflections (indexed by "use case")
            for namespace_template in episodic_config.get("reflectionNamespaces", []):
                namespace = namespace_template.format(actorId=actor_id)
                reflections = self.client.retrieve_memories(
                    memory_id=self.memory_id,
                    namespace=namespace,
                    query=user_query,  # Reflections indexed by use case
                    top_k=2
                )
                
                for reflection in reflections:
                    if isinstance(reflection, dict):
                        content = reflection.get('content', {})
                        if isinstance(content, dict):
                            text = content.get('text', '').strip()
                            if text:
                                all_context.append(f"[REFLECTION] {text}")
            
            # Inject context into query
            if all_context:
                context_text = "\n".join(all_context)
                original_text = messages[-1]["content"][0]["text"]
                messages[-1]["content"][0]["text"] = (
                    f"Past Experience:\n{context_text}\n\nCurrent Query: {original_text}"
                )
                logger.info(f"Retrieved {len(all_context)} episodes/reflections")
                
        except Exception as e:
            logger.error(f"Failed to retrieve episodes: {e}")
    
    def save_meeting_interaction(self, event: AfterInvocationEvent):
        """Save meeting interaction for episode extraction."""
        try:
            messages = event.agent.messages
            if len(messages) < 2 or messages[-1]["role"] != "assistant":
                return
            
            # Collect the full interaction including tool uses
            interaction_messages = []
            for msg in messages:
                role = msg["role"].upper()
                content = msg["content"]
                
                if isinstance(content, list):
                    for item in content:
                        if "text" in item:
                            interaction_messages.append((item["text"], role))
                        elif "toolUse" in item:
                            # Include tool usage for better episode extraction
                            tool_info = item["toolUse"]
                            tool_text = f"[TOOL: {tool_info.get('name', 'unknown')}]"
                            interaction_messages.append((tool_text, "TOOL"))
                        elif "toolResult" in item:
                            result = item["toolResult"].get("content", [{}])[0].get("text", "")
                            interaction_messages.append((f"[RESULT: {result[:200]}]", "TOOL"))
            
            if interaction_messages:
                actor_id = event.agent.state.get("actor_id")
                session_id = event.agent.state.get("session_id")
                
                if not actor_id or not session_id:
                    logger.warning("Missing actor_id or session_id")
                    return
                
                # Save event - AgentCore will automatically detect episode completion
                self.client.create_event(
                    memory_id=self.memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=interaction_messages
                )
                logger.info("Saved meeting interaction for episode extraction")
                
        except Exception as e:
            logger.error(f"Failed to save interaction: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register episodic memory hooks."""
        registry.add_callback(MessageAddedEvent, self.retrieve_episodes_and_reflections)
        registry.add_callback(AfterInvocationEvent, self.save_meeting_interaction)
        logger.info("Episodic memory hooks registered")

## Passo 5: Criar Agente de Notas de Reunião

In [42]:
episodic_hooks = EpisodicMemoryHooks(memory_id, client)

meeting_agent = Agent(
    hooks=[episodic_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[capture_action_item, identify_decision, summarize_discussion, track_followup],
    state={"actor_id": PARTICIPANT_ID, "session_id": SESSION_ID},
    system_prompt="""You are an expert meeting assistant with memory of past meetings.

Your role:
- Help facilitate productive meetings by tracking decisions and action items
- Use past meeting episodes to provide relevant context and history
- Apply reflections about what works well for different teams and participants
- Remember participant preferences and communication styles

When you see [PAST EPISODE] context, use it to inform your responses.
When you see [REFLECTION] context, apply those learned patterns.

Always:
1. Listen for key decisions and action items
2. Reference relevant past meetings when helpful
3. Track follow-ups on previous action items
4. Summarize discussions clearly and concisely""")

print("✅ Meeting notes agent created with episodic memory")

2026-03-24 20:00:44,753 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-24 20:00:44,913 - INFO - Episodic memory hooks registered


✅ Meeting notes agent created with episodic memory


## Passo 6: Alimentar Episódios de Reuniões Anteriores

Vamos adicionar algumas sessões de reuniões anteriores para demonstrar a episodic memory.

In [43]:
# Seed with previous meeting sessions
past_sessions = [
    # Session 1: Sprint planning meeting
    ("Let's plan the Q3 sprint. We need to prioritize features.", "USER"),
    ("I'll help capture the key decisions and action items.", "ASSISTANT"),
    (
        "We should focus on the user authentication feature first. It's blocking other work.",
        "USER",
    ),
    ("[TOOL: identify_decision]", "TOOL"),
    ("[RESULT: 📌 DECISION RECORDED: Prioritize user authentication feature for Q3 sprint]", "TOOL"),
    (
        "Noted. User authentication is the priority. Who will lead this?",
        "ASSISTANT",
    ),
    ("Sarah can handle the authentication work. She has experience with OAuth.", "USER"),
    ("[TOOL: capture_action_item]", "TOOL"),
    ("[RESULT: ✅ ACTION ITEM CAPTURED: Implement user authentication - Sarah - Due: End of sprint]", "TOOL"),
    (
        "Perfect! I've captured that Sarah will implement user authentication by end of sprint.",
        "ASSISTANT",
    ),
    # Session 2: Budget review meeting
    (
        "We need to discuss the Q3 marketing budget. Costs are higher than expected.",
        "USER",
    ),
    ("Let me help track this discussion.", "ASSISTANT"),
    ("[TOOL: summarize_discussion]", "TOOL"),
    ("[RESULT: 📝 DISCUSSION SUMMARY: Q3 marketing budget - costs exceeding projections]", "TOOL"),
    (
        "I propose we increase the budget by 15% to account for the new campaigns.",
        "USER",
    ),
    ("[TOOL: identify_decision]", "TOOL"),
    ("[RESULT: 📌 DECISION RECORDED: Approved Q3 marketing budget increase of 15%]", "TOOL"),
    (
        "Decision captured. Is there a follow-up needed?",
        "ASSISTANT",
    ),
    ("Yes, Mike needs to update the financial forecast by end of week.", "USER"),
    ("[TOOL: capture_action_item]", "TOOL"),
    ("[RESULT: ✅ ACTION ITEM CAPTURED: Update financial forecast - Mike - Due: End of week]", "TOOL"),
]

try:
    client.create_event(
        memory_id=memory_id,
        actor_id=PARTICIPANT_ID,
        session_id="seed_session_001",
        messages=past_sessions
    )
    print("✅ Seeded past meeting episodes")
    print("⏳ Note: Episode extraction happens in background (~1 minute)")
    print("⏳ Note: Reflection extraction takes 10-15 minutes after episodes are stored")
except Exception as e:
    print(f"⚠️ Error seeding history: {e}")

# Wait for episode extraction to complete before running tests
import time

print("\n⏳ Waiting for episode extraction to complete...")
print("   (Episode extraction is asynchronous and typically takes 1-3 minutes)")

max_wait = 300  # Maximum 5 minutes
poll_interval = 30  # Check every 30 seconds
elapsed = 0
episodes_found = False

# Get namespace for polling
_episodic_ns = get_namespaces(client, memory_id).get("EPISODIC", {})
_ns_templates = _episodic_ns.get("namespaces", [])

while elapsed < max_wait:
    time.sleep(poll_interval)
    elapsed += poll_interval
    
    # Check if episodes have been extracted
    for ns_template in _ns_templates:
        ns = ns_template.format(actorId=PARTICIPANT_ID)
        try:
            results = client.retrieve_memories(
                memory_id=memory_id,
                namespace=ns,
                query="sprint planning budget decisions",
                top_k=1
            )
            if results:
                episodes_found = True
                break
        except Exception:
            pass
    
    if episodes_found:
        print(f"✅ Episodes detected after {elapsed} seconds!")
        break
    else:
        print(f"   ⏳ {elapsed}s elapsed - no episodes yet, continuing to wait...")

if not episodes_found:
    print(f"⚠️ No episodes found after {max_wait}s. Tests will proceed but may not have past context.")
    print("   This is expected if the conversation was too short for episode detection.")
else:
    print("✅ Episodes are ready for retrieval. Proceeding with tests.")

2026-03-24 20:00:45,752 - INFO - Created event: 0000001774393244925#0b57efc4


✅ Seeded past meeting episodes
⏳ Note: Episode extraction happens in background (~1 minute)
⏳ Note: Reflection extraction takes 10-15 minutes after episodes are stored

⏳ Waiting for episode extraction to complete...
   (Episode extraction is asynchronous and typically takes 1-3 minutes)


2026-03-24 20:01:16,614 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


   ⏳ 30s elapsed - no episodes yet, continuing to wait...


2026-03-24 20:01:47,092 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


...

   ⏳ 240s elapsed - no episodes yet, continuing to wait...


2026-03-24 20:05:20,140 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


   ⏳ 270s elapsed - no episodes yet, continuing to wait...


2026-03-24 20:05:50,583 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


   ⏳ 300s elapsed - no episodes yet, continuing to wait...
⚠️ No episodes found after 300s. Tests will proceed but may not have past context.
   This is expected if the conversation was too short for episode detection.


## Passo 7: Testar Cenários de Reunião

O agente agora deve aproveitar episódios e reflexões anteriores.

In [44]:
# Test 1: Follow-up on previous decision - should reference past episode
response1 = meeting_agent("Let's revisit the Q3 sprint priorities we discussed last week. What was decided?")
print(f"Agent: {response1}")

2026-03-24 20:05:51,027 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


I don't have access to a record of your meeting from last week, so I can't retrieve the specific Q3 sprint priorities that were discussed or the decisions that were made.

However, I can help you in a couple of ways:

1. **If you can recall the key points**: You could share what you remember about the Q3 sprint priorities discussion, and I can help you document or organize that information formally.

2. **Going forward**: I can help you track and document decisions and action items from today's meeting so you have a clear record for future reference.

3. **Current status check**: If there were specific action items from last week's meeting that need follow-up, I can help you track their status using the tools available to me.

Would you like to:
- Share the priorities you remember so we can document them?
- Check on the status of any action items from that previous meeting?
- Start fresh with a new Q3 sprint priorities discussion today?

What would be most helpful for you?

2026-03-24 20:05:54,534 - INFO - Created event: 0000001774393554161#8a515710
2026-03-24 20:05:54,534 - INFO - Saved meeting interaction for episode extraction


Agent: I don't have access to a record of your meeting from last week, so I can't retrieve the specific Q3 sprint priorities that were discussed or the decisions that were made.

However, I can help you in a couple of ways:

1. **If you can recall the key points**: You could share what you remember about the Q3 sprint priorities discussion, and I can help you document or organize that information formally.

2. **Going forward**: I can help you track and document decisions and action items from today's meeting so you have a clear record for future reference.

3. **Current status check**: If there were specific action items from last week's meeting that need follow-up, I can help you track their status using the tools available to me.

Would you like to:
- Share the priorities you remember so we can document them?
- Check on the status of any action items from that previous meeting?
- Start fresh with a new Q3 sprint priorities discussion today?

What would be most helpful for you?



In [45]:
# Test 2: Action item check - should retrieve past action items
response2 = meeting_agent("Did we assign someone to handle the user authentication feature?")
print(f"Agent: {response2}")

2026-03-24 20:05:54,967 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


I don't have access to records from your previous meetings, so I can't tell you whether someone was assigned to handle the user authentication feature or who that person might be.

However, I can help you:

1. **Check on this now**: If you remember who might have been assigned, I can help you track the status of that action item to see where it stands.

2. **Reassign if needed**: If the assignment has changed or wasn't clearly documented, we can capture a new action item with clear ownership and a due date.

3. **Document for the future**: Going forward, I'll be able to track all assignments and decisions from your meetings so you have a complete record.

Do you want to:
- Look up who might be responsible and check their status on this task?
- Create a fresh action item for the user authentication feature with clear ownership?
- Review your team's current workload to determine the best person for this?

What would work best for you?

2026-03-24 20:05:58,609 - INFO - Created event: 0000001774393558250#57024488
2026-03-24 20:05:58,610 - INFO - Saved meeting interaction for episode extraction


Agent: I don't have access to records from your previous meetings, so I can't tell you whether someone was assigned to handle the user authentication feature or who that person might be.

However, I can help you:

1. **Check on this now**: If you remember who might have been assigned, I can help you track the status of that action item to see where it stands.

2. **Reassign if needed**: If the assignment has changed or wasn't clearly documented, we can capture a new action item with clear ownership and a due date.

3. **Document for the future**: Going forward, I'll be able to track all assignments and decisions from your meetings so you have a complete record.

Do you want to:
- Look up who might be responsible and check their status on this task?
- Create a fresh action item for the user authentication feature with clear ownership?
- Review your team's current workload to determine the best person for this?

What would work best for you?



In [46]:
# Test 3: Budget follow-up - should reference past budget discussion
response3 = meeting_agent("What was the outcome of the Q3 marketing budget discussion?")
print(f"Agent: {response3}")

2026-03-24 20:06:00,142 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


I don't have access to records from your previous meetings, so I'm unable to tell you what was decided regarding the Q3 marketing budget.

To help you move forward, I can:

1. **Recreate the record**: If you can recall the outcome of that discussion, I can help you document it now for future reference.

2. **Track follow-ups**: If there were action items or decisions from that marketing budget discussion that need attention, I can help you track their status.

3. **Document today's discussion**: If you're revisiting this topic in a current meeting, I can capture any new decisions or action items that come out of today's conversation.

Do you have notes or remember the key points from that discussion? If so, I'd be happy to help you formalize and track them. Or would you like to start fresh with a new discussion on the Q3 marketing budget?

2026-03-24 20:06:03,103 - INFO - Created event: 0000001774393562743#04daa713
2026-03-24 20:06:03,104 - INFO - Saved meeting interaction for episode extraction


Agent: I don't have access to records from your previous meetings, so I'm unable to tell you what was decided regarding the Q3 marketing budget.

To help you move forward, I can:

1. **Recreate the record**: If you can recall the outcome of that discussion, I can help you document it now for future reference.

2. **Track follow-ups**: If there were action items or decisions from that marketing budget discussion that need attention, I can help you track their status.

3. **Document today's discussion**: If you're revisiting this topic in a current meeting, I can capture any new decisions or action items that come out of today's conversation.

Do you have notes or remember the key points from that discussion? If so, I'd be happy to help you formalize and track them. Or would you like to start fresh with a new discussion on the Q3 marketing budget?



In [47]:
# Test 4: New meeting with multiple actions
response4 = meeting_agent("""We're having a product launch planning meeting. Key points:
- Launch date: November 15th
- Marketing team needs 2 weeks prep time
- Sarah will coordinate with vendors
- Mike needs to finalize pricing by next Friday

Can you capture the decisions and action items?""")
print(f"Agent: {response4}")

2026-03-24 20:06:03,491 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes



Tool #1: identify_decision

Tool #2: identify_decision

Tool #3: capture_action_item

Tool #4: capture_action_item
Perfect! I've captured everything from your product launch planning meeting:

**📌 Decisions Made:**
- **Product launch date: November 15th** - This is your official launch date
- **Marketing team needs 2 weeks of preparation time** - They'll need to begin their campaign prep by approximately November 1st

**✅ Action Items:**
- **Sarah**: Coordinate with vendors for product launch (Due: November 15th)
- **Mike**: Finalize pricing for product launch (Due: Next Friday)

**Quick note**: You may want to flag that Mike's pricing deadline (next Friday) needs to happen well before November 1st so the marketing team has the final pricing details during their 2-week prep window. This ensures marketing materials have accurate pricing information.

Is there anything else you'd like to capture from this meeting, or any other considerations for the launch planning?

2026-03-24 20:06:09,242 - INFO - Created event: 0000001774393568890#20e72e15
2026-03-24 20:06:09,243 - INFO - Saved meeting interaction for episode extraction


Agent: Perfect! I've captured everything from your product launch planning meeting:

**📌 Decisions Made:**
- **Product launch date: November 15th** - This is your official launch date
- **Marketing team needs 2 weeks of preparation time** - They'll need to begin their campaign prep by approximately November 1st

**✅ Action Items:**
- **Sarah**: Coordinate with vendors for product launch (Due: November 15th)
- **Mike**: Finalize pricing for product launch (Due: Next Friday)

**Quick note**: You may want to flag that Mike's pricing deadline (next Friday) needs to happen well before November 1st so the marketing team has the final pricing details during their 2-week prep window. This ensures marketing materials have accurate pricing information.

Is there anything else you'd like to capture from this meeting, or any other considerations for the launch planning?



In [48]:
# Test 5: Pattern recognition - agent should remember participant preferences
response5 = meeting_agent("Sarah wants to discuss technical architecture for the new feature. What format works best?")
print(f"Agent: {response5}")

2026-03-24 20:06:09,658 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


That's a great question! To give you the best recommendation on discussion format, it would help to know a bit more:

1. **Who needs to be involved?** (e.g., just the development team, cross-functional stakeholders, leadership?)

2. **What's the scope?** (Is this a complex architecture that needs deep technical discussion, or more of a high-level overview?)

3. **What's the goal?** (Are you looking to get buy-in, make decisions, brainstorm solutions, or document the final design?)

4. **Timeline?** (Is this urgent or can it be scheduled strategically?)

**Common formats that work well for technical architecture discussions:**
- **Whiteboarding session** - Great for real-time ideation and drawing diagrams
- **Architecture review meeting** - Structured format for presenting and getting feedback
- **Working session** - Hands-on discussion with developers to dive deep into implementation
- **Async documentation + brief sync** - For complex topics, share detailed design docs first, then mee

2026-03-24 20:06:13,974 - INFO - Created event: 0000001774393573578#f0497f4f
2026-03-24 20:06:13,975 - INFO - Saved meeting interaction for episode extraction


Agent: That's a great question! To give you the best recommendation on discussion format, it would help to know a bit more:

1. **Who needs to be involved?** (e.g., just the development team, cross-functional stakeholders, leadership?)

2. **What's the scope?** (Is this a complex architecture that needs deep technical discussion, or more of a high-level overview?)

3. **What's the goal?** (Are you looking to get buy-in, make decisions, brainstorm solutions, or document the final design?)

4. **Timeline?** (Is this urgent or can it be scheduled strategically?)

**Common formats that work well for technical architecture discussions:**
- **Whiteboarding session** - Great for real-time ideation and drawing diagrams
- **Architecture review meeting** - Structured format for presenting and getting feedback
- **Working session** - Hands-on discussion with developers to dive deep into implementation
- **Async documentation + brief sync** - For complex topics, share detailed design docs first, t

In [49]:
# Test 6: Completely new topic - no past context
response6 = meeting_agent("We need to discuss the company's sustainability initiative for the first time. Let's brainstorm ideas.")
print(f"Agent: {response6}")

2026-03-24 20:06:14,361 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes


Great! I'd be happy to help facilitate a brainstorming session on your company's sustainability initiative. To make this most productive, let me ask a few clarifying questions:

1. **Scope & Focus**: Are you looking at sustainability broadly (environmental, social, economic), or focusing on specific areas like:
   - Carbon footprint reduction?
   - Waste management?
   - Energy efficiency?
   - Supply chain sustainability?
   - Employee/community impact?

2. **Business Context**: 
   - What industry/type of company are you in?
   - Are there specific pain points or opportunities you've already identified?

3. **Goals**:
   - Are you aiming for internal practices, customer-facing initiatives, or both?
   - Any existing commitments or timelines (e.g., net-zero by 2030)?

4. **Participants**: 
   - Who's attending this brainstorm? (leadership, cross-functional teams, sustainability experts?)
   - What perspectives do you want to bring to the table?

5. **Outcomes**:
   - What do you want 

2026-03-24 20:06:18,759 - INFO - Created event: 0000001774393578406#9afa4b69
2026-03-24 20:06:18,760 - INFO - Saved meeting interaction for episode extraction


Agent: Great! I'd be happy to help facilitate a brainstorming session on your company's sustainability initiative. To make this most productive, let me ask a few clarifying questions:

1. **Scope & Focus**: Are you looking at sustainability broadly (environmental, social, economic), or focusing on specific areas like:
   - Carbon footprint reduction?
   - Waste management?
   - Energy efficiency?
   - Supply chain sustainability?
   - Employee/community impact?

2. **Business Context**: 
   - What industry/type of company are you in?
   - Are there specific pain points or opportunities you've already identified?

3. **Goals**:
   - Are you aiming for internal practices, customer-facing initiatives, or both?
   - Any existing commitments or timelines (e.g., net-zero by 2030)?

4. **Participants**: 
   - Who's attending this brainstorm? (leadership, cross-functional teams, sustainability experts?)
   - What perspectives do you want to bring to the table?

5. **Outcomes**:
   - What do yo

## Passo 8: Verificar Armazenamento de Episódios

In [50]:
print("\n📚 Episodic Memory Summary:")
print("=" * 50)

episodic_config = get_namespaces(client, memory_id).get("EPISODIC", {})

# Check episodes
for namespace_template in episodic_config.get("namespaces", []):
    namespace = namespace_template.format(actorId=PARTICIPANT_ID)
    
    try:
        episodes = client.retrieve_memories(
            memory_id=memory_id,
            namespace=namespace,
            query="meeting decisions action items",
            top_k=5
        )
        
        print(f"\nEPISODES ({len(episodes)} found):")
        for i, episode in enumerate(episodes, 1):
            if isinstance(episode, dict):
                content = episode.get('content', {})
                if isinstance(content, dict):
                    text = content.get('text', '')[:200] + "..."
                    print(f"  {i}. {text}")
                
    except Exception as e:
        print(f"Error retrieving episodes: {e}")

# Check reflections
for namespace_template in episodic_config.get("reflectionNamespaces", []):
    namespace = namespace_template.format(actorId=PARTICIPANT_ID)
    
    try:
        reflections = client.retrieve_memories(
            memory_id=memory_id,
            namespace=namespace,
            query="meeting patterns effectiveness",
            top_k=3
        )
        
        print(f"\nREFLECTIONS ({len(reflections)} found):")
        for i, reflection in enumerate(reflections, 1):
            if isinstance(reflection, dict):
                content = reflection.get('content', {})
                if isinstance(content, dict):
                    text = content.get('text', '')[:200] + "..."
                    print(f"  {i}. {text}")
                
    except Exception as e:
        print(f"Error retrieving reflections: {e}")

print("\n" + "=" * 50)


📚 Episodic Memory Summary:


2026-03-24 20:06:19,938 - INFO - Retrieved 0 memories from namespace: meetings/actor/participant_001/episodes



EPISODES (0 found):



## Principais Conclusões

1. **Episodic memory captura sequências de interação**, não apenas fatos
2. **Reflections emergem automaticamente** da análise de múltiplos episódios
3. **Inclua resultados de ferramentas** nos eventos para uma extração de episódios mais rica
4. **Consulte por intenção** para episódios, **por caso de uso** para reflections
5. **A extração de episódios é assíncrona** (~1 minuto após o fim da conversa)

## Limpeza

In [51]:
# Clean up all resources created during notebook execution
print("=== Limpeza de Recursos ===")

# Step 1: Delete events from the seed session
try:
    seed_events = client.list_events(
        memory_id=memory_id,
        actor_id=PARTICIPANT_ID,
        session_id="seed_session_001"
    )
    if seed_events:
        for event in seed_events:
            event_id = event.get('eventId') or event.get('id')
            if event_id:
                client.delete_event(
                    memory_id=memory_id,
                    event_id=event_id,
                    session_id="seed_session_001",
                    actor_id=PARTICIPANT_ID
                )
        print(f"✅ Deleted {len(seed_events)} events from seed session")
except Exception as e:
    print(f"⚠️ Could not delete seed events: {e}")

# Step 2: Delete events from the agent test session
try:
    test_events = client.list_events(
        memory_id=memory_id,
        actor_id=PARTICIPANT_ID,
        session_id=SESSION_ID
    )
    if test_events:
        for event in test_events:
            event_id = event.get('eventId') or event.get('id')
            if event_id:
                client.delete_event(
                    memory_id=memory_id,
                    event_id=event_id,
                    session_id=SESSION_ID,
                    actor_id=PARTICIPANT_ID
                )
        print(f"✅ Deleted {len(test_events)} events from test session")
except Exception as e:
    print(f"⚠️ Could not delete test events: {e}")

# Step 3: Delete the Memory resource (removes memory, strategies, episodes, reflections, and all records)
try:
    client.delete_memory_and_wait(memory_id=memory_id)
    print(f"✅ Deleted memory resource: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")

print("=== Limpeza concluída ===")

=== Limpeza de Recursos ===


2026-03-24 20:06:20,211 - INFO - Retrieved total of 1 events


✅ Deleted 1 events from seed session


2026-03-24 20:06:20,980 - INFO - Retrieved total of 6 events


✅ Deleted 6 events from test session


2026-03-24 20:06:23,054 - INFO - Deleted memory: MeetingNotesEpisodicMemory-3azWF78Tgo
2026-03-24 20:06:23,055 - INFO - Initiated deletion of memory MeetingNotesEpisodicMemory-3azWF78Tgo
2026-03-24 20:06:54,220 - INFO - Memory MeetingNotesEpisodicMemory-3azWF78Tgo successfully deleted (took 30 seconds)


✅ Deleted memory resource: MeetingNotesEpisodicMemory-3azWF78Tgo
=== Limpeza concluída ===
